# Visual-Language Navigation for Legged Robots using RGB Images
## CS 691 Course Project 

### Imports 

In [ ]:
import os, json, pickle, functools, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import clip
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.spatial import KDTree
from skimage.draw import line_aa
from skimage.draw import line as sk_line
from datasets import load_dataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch  : {torch.__version__}")
print(f"device : {DEVICE}")

In [ ]:
CFG = {
    "embodiment":          "Legged Robot",
    "clip_model":          "ViT-B/32",
    "hidden_dim":          256,
    "dropout":             0.15,
    "weight_decay":        5e-4,
    "lr":                  1e-4,
    "batch_size":          32,
    "epochs":              300,
    "patience":            30,     # early stopping patience
    "goal_weight":         1.5,
    "endpoint_weight":     1.5,
    "val_split":           0.2,
    "seed":                42,
    "penalty_tsv":         "./category_penalty.tsv",
    "m2f_config":          "./mask2former_config.json",
    "bad_score_threshold": 3234.75,
    "penalty_dist_thr":    35,
}

torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])
random.seed(CFG["seed"]) 

## Data Setup

In [ ]:
ALL_SAMPLES_PKL = "./samples_all.pkl"

if os.path.exists(ALL_SAMPLES_PKL):
    print("Loading from local pkl...")
    with open(ALL_SAMPLES_PKL, "rb") as f:
        saved = pickle.load(f)
    val_split_samples = saved["val_split"]
    print(f"NaviTrace val split : {len(val_split_samples)} samples")
else:
    print("Loading NaviTrace from HuggingFace...")
    dataset = load_dataset("leggedrobotics/navitrace")
    print(f"Available splits : {list(dataset.keys())}")

    val_split_samples, skipped = [], 0
    for s in tqdm(list(dataset["validation"]), desc="Filtering"):
        gt = s["ground_truth"].get(CFG["embodiment"])
        if gt is not None and len(gt) > 0:
            val_split_samples.append(s)
        else:
            skipped += 1
    print(f"Kept={len(val_split_samples)}  Skipped={skipped}")

    with open(ALL_SAMPLES_PKL, "wb") as f:
        pickle.dump({"val_split": val_split_samples}, f)

# Compute N waypoints from median trace length
trace_lengths = []
for s in val_split_samples:
    for t in s["ground_truth"][CFG["embodiment"]]:
        trace_lengths.append(len(t))
N = int(np.median(trace_lengths))
print(f"N = {N} waypoints")

# Fixed random split
random.seed(CFG["seed"])
random.shuffle(val_split_samples)

n_our_val     = int(len(val_split_samples) * CFG["val_split"])
val_samples   = val_split_samples[:n_our_val]
train_samples = val_split_samples[n_our_val:]

print(f"Train : {len(train_samples)}")
print(f"Val   : {len(val_samples)}")


## Preprocessing 

Let's make the image square, convert GT trace, and intperpolate trace for our regression output 

In [ ]:
def pad_to_square(image):
    w, h   = image.size
    max_s  = max(w, h)
    padded = Image.new("RGB", (max_s, max_s), (0, 0, 0))
    x_off  = (max_s - w) // 2
    y_off  = (max_s - h) // 2
    padded.paste(image, (x_off, y_off))
    return padded, (x_off/max_s, y_off/max_s, w/max_s, h/max_s)

def adjust_trace(trace_px, orig_w, orig_h, xof, yof, sx, sy):
    t = np.array(trace_px, dtype=float)
    t[:,0] = (t[:,0] / orig_w) * sx + xof
    t[:,1] = (t[:,1] / orig_h) * sy + yof
    return t.astype(np.float32)

def interpolate_trace(trace: np.ndarray, N: int) -> np.ndarray:
    if len(trace) == 0: return np.zeros((N, 2), dtype=np.float32)
    if len(trace) == 1: return np.tile(trace[0], (N, 1)).astype(np.float32)
    diffs  = np.diff(trace, axis=0)
    dists  = np.sqrt((diffs**2).sum(axis=1))
    cumlen = np.concatenate([[0], np.cumsum(dists)])
    total  = cumlen[-1]
    if total == 0: return np.tile(trace[0], (N, 1)).astype(np.float32)
    t_old = cumlen / total
    t_new = np.linspace(0, 1, N)
    return np.stack([
        np.interp(t_new, t_old, trace[:,0]),
        np.interp(t_new, t_old, trace[:,1])
    ], axis=1).astype(np.float32)

def pred_padded_to_pixel(pred_norm, orig_w, orig_h):
    max_s  = max(orig_w, orig_h)
    x_off  = (max_s - orig_w) / 2
    y_off  = (max_s - orig_h) / 2
    p      = pred_norm.copy()
    p[:,0] = np.clip(p[:,0] * max_s - x_off, 0, orig_w - 1)
    p[:,1] = np.clip(p[:,1] * max_s - y_off, 0, orig_h - 1)
    return p.tolist()


Now, normalize the goal targets

In [ ]:
gt_goals = []
for s in train_samples:
    gt_traces = s["ground_truth"].get(CFG["embodiment"])
    if not gt_traces: continue
    img = s["image"]
    orig_w, orig_h = img.size
    _, (xof, yof, sx, sy) = pad_to_square(img)
    for t in gt_traces:
        gt_norm = adjust_trace(np.array(t), orig_w, orig_h, xof, yof, sx, sy)
        gt_goals.append(gt_norm[-1])    

gt_goals  = np.array(gt_goals)
goal_mean = torch.tensor(gt_goals.mean(axis=0), dtype=torch.float32)
goal_std  = torch.tensor(gt_goals.std(axis=0),  dtype=torch.float32) 
 
print(f"Goal mean : {goal_mean.numpy().round(4)}, Goal std  : {goal_std.numpy().round(4)}")

## Clip Embeddings 

In [ ]:
EMBED_TRAIN = "./train_embeddings_simple.pt"
EMBED_VAL   = "./val_embeddings_simple.pt"

if os.path.exists(EMBED_TRAIN) and os.path.exists(EMBED_VAL):
    print("Loading cached embeddings...")
    train_emb         = torch.load(EMBED_TRAIN, weights_only=False)
    val_emb           = torch.load(EMBED_VAL,   weights_only=False)
    train_img_emb     = train_emb["img_emb"]
    train_txt_emb     = train_emb["txt_emb"]
    train_samples_ref = train_emb["samples"]
    val_img_emb       = val_emb["img_emb"]
    val_txt_emb       = val_emb["txt_emb"]
else:
    print("Extracting CLIP embeddings (run once, cached after)...")
    clip_model, clip_preprocess = clip.load(CFG["clip_model"], device=DEVICE)
    clip_model.eval()
    for p in clip_model.parameters():
        p.requires_grad = False   # freeze all CLIP weights

    class EmbedDS(Dataset):
        def __init__(self, samps):
            self.samps = samps
        def __len__(self): return len(self.samps)
        def __getitem__(self, idx):
            s   = self.samps[idx]
            img = s["image"].convert("RGB")
            padded, _ = pad_to_square(img)
            img_t = clip_preprocess(padded)
            txt_t = clip.tokenize([s["task"]], truncate=True)[0]
            return img_t, txt_t

    @torch.no_grad()
    def extract(samps):
        ds  = EmbedDS(samps)
        ldr = DataLoader(ds, batch_size=64, shuffle=False, num_workers=0)
        imgs, txts = [], []
        for i_t, t_t in tqdm(ldr, desc="Extracting"):
            i_f = F.normalize(clip_model.encode_image(i_t.to(DEVICE)).float(), dim=-1)
            t_f = F.normalize(clip_model.encode_text(t_t.to(DEVICE)).float(),  dim=-1)
            imgs.append(i_f.cpu()); txts.append(t_f.cpu())
        return torch.cat(imgs), torch.cat(txts)

    train_img_emb, train_txt_emb = extract(train_samples)
    val_img_emb,   val_txt_emb   = extract(val_samples)
    train_samples_ref = list(train_samples)

    torch.save({"img_emb": train_img_emb, "txt_emb": train_txt_emb,
                "samples": train_samples_ref}, EMBED_TRAIN)
    torch.save({"img_emb": val_img_emb, "txt_emb": val_txt_emb}, EMBED_VAL)
    print("Saved embeddings")

train_joint = torch.cat([train_img_emb, train_txt_emb], dim=-1)  # (n, 1024)
val_joint   = torch.cat([val_img_emb,   val_txt_emb],   dim=-1)
print(f"Train joint : {train_joint.shape}, Val   joint : {val_joint.shape}")

## Task Heads

In [ ]:
class NavigationMLP(nn.Module):
  
    def __init__(self, N=N):
        super().__init__()
        self.N = N
        self.shared = nn.Sequential(
            nn.Linear(1024, CFG["hidden_dim"]),
            nn.ReLU(),
            nn.Dropout(CFG["dropout"]),
        )
        self.goal_head = nn.Sequential(
            nn.Linear(CFG["hidden_dim"], 64),
            nn.ReLU(),
            nn.Linear(64, 2)
            # No sigmoid -- outputs in normalized space
        )
        self.trace_head = nn.Sequential(
            nn.Linear(CFG["hidden_dim"] + 2, 128),
            nn.ReLU(),
            nn.Dropout(CFG["dropout"]),
            nn.Linear(128, N * 2),
            nn.Sigmoid()
        )

    def forward(self, x):
        shared   = self.shared(x)
        goal     = self.goal_head(shared)
        trace_in = torch.cat([shared, goal.detach()], dim=-1)
        trace    = self.trace_head(trace_in)
        return goal, trace.view(-1, self.N, 2)


model  = NavigationMLP(N=N).to(DEVICE)
params = sum(p.numel() for p in model.parameters())
print(params)

## Training 

For the Loss, we normalize and compute loss against GT trace w/ min 

In [ ]:
def nav_loss(goal_pred_norm, trace_pred, sample, device):

    gt_traces = sample["ground_truth"].get(CFG["embodiment"])
    if gt_traces is None or len(gt_traces) == 0:
        return torch.tensor(0.0, requires_grad=True, device=device)

    img = sample["image"]
    orig_w, orig_h = img.size
    _, (xof, yof, sx, sy) = pad_to_square(img)
    gm = goal_mean.to(device)
    gs = goal_std.to(device)

    best_loss = None
    for gt_px in gt_traces:
        gt_norm  = adjust_trace(
            np.array(gt_px, dtype=float), orig_w, orig_h, xof, yof, sx, sy
        )
        gt_fixed = torch.tensor(
            interpolate_trace(gt_norm, trace_pred.shape[0]),
            dtype=torch.float32, device=device
        )
        gt_goal        = gt_fixed[-1]
        gt_goal_normed = (gt_goal - gm) / (gs + 1e-8)

        # MSE on normalized goal
        loss_goal  = F.mse_loss(goal_pred_norm, gt_goal_normed)

        # MSE on full trace
        loss_trace = F.mse_loss(trace_pred, gt_fixed)

        # Extra MSE on last waypoint
        loss_end   = F.mse_loss(trace_pred[-1], gt_goal)

        total = (
            CFG["goal_weight"]     * loss_goal  + loss_trace                           +
            CFG["endpoint_weight"] * loss_end
        )
        if best_loss is None or total.item() < best_loss.item():
            best_loss = total

    return best_loss


def denorm_goal(goal_norm_np): 
    g = goal_norm_np * goal_std.numpy() + goal_mean.numpy()
    return np.clip(g, 0, 1)
 

We use AdamW optimizer and scheduler. Early stopping when we do not see any improvment 

In [ ]:
CKPT = "./best_mlp_simple.pt"

if os.path.exists(CKPT):
    print(f"Checkpoint found - loading {CKPT}")
    model.load_state_dict(torch.load(CKPT, map_location=DEVICE, weights_only=True))
    train_losses, val_losses = [], []
else:
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CFG["lr"],
        weight_decay=CFG["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG["epochs"]
    )

    train_losses, val_losses = [], []
    best_val, best_epoch     = float("inf"), 0
    no_improve               = 0
    n_train = len(train_samples_ref)
    n_val_s = len(val_samples)

    for epoch in range(1, CFG["epochs"] + 1):
        model.train()
        perm  = torch.randperm(n_train)
        tloss = 0.0
        steps = 0

        for start in range(0, n_train, CFG["batch_size"]):
            batch_idx = perm[start: start + CFG["batch_size"]].tolist()
            x_b = torch.cat([
                train_img_emb[batch_idx].to(DEVICE),
                train_txt_emb[batch_idx].to(DEVICE)
            ], dim=-1)

            optimizer.zero_grad()
            goal_preds, trace_preds = model(x_b)

            batch_loss = torch.tensor(0.0, device=DEVICE)
            for j, si in enumerate(batch_idx):
                loss = nav_loss(
                    goal_preds[j], trace_preds[j],
                    train_samples_ref[si], DEVICE
                )
                batch_loss = batch_loss + loss

            (batch_loss / len(batch_idx)).backward()
            optimizer.step()
            tloss += (batch_loss / len(batch_idx)).item()
            steps += 1

        model.eval()
        vloss  = 0.0
        vsteps = 0
        with torch.no_grad():
            for start in range(0, n_val_s, CFG["batch_size"]):
                end = min(start + CFG["batch_size"], n_val_s)
                x_b = torch.cat([
                    val_img_emb[start:end].to(DEVICE),
                    val_txt_emb[start:end].to(DEVICE)
                ], dim=-1)
                gp, tp = model(x_b)
                for j in range(end - start):
                    loss    = nav_loss(gp[j], tp[j], val_samples[start+j], DEVICE)
                    vloss  += loss.item()
                    vsteps += 1

        tloss /= steps
        vloss /= vsteps
        scheduler.step()
        train_losses.append(tloss)
        val_losses.append(vloss)

        #  Early stopping 
        if vloss < best_val:
            best_val, best_epoch = vloss, epoch
            no_improve = 0
            torch.save(model.state_dict(), CKPT)
        else:
            no_improve += 1

        if epoch % 10 == 0:
            print(f"Epoch {epoch:3d}/{CFG['epochs']}  "
                  f"train={tloss:.5f}  val={vloss:.5f}  "
                  f"best={best_val:.5f} at epoch {best_epoch}  "
                  f"patience={no_improve}/{CFG['patience']}")

        if no_improve >= CFG["patience"]:
            print(f"Early stopping at epoch {epoch}")
            break

    if os.path.exists(CKPT):
        model.load_state_dict(torch.load(CKPT, map_location=DEVICE, weights_only=True))
        print(f"Training complete. Best val={best_val:.5f} at epoch {best_epoch}")
    else:
        print("No checkpoint saved")

if train_losses:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(train_losses, label="train")
    ax.plot(val_losses,   label="val")
    best_ep = int(np.argmin(val_losses))
    ax.axvline(best_ep, color="red", ls="--", lw=1,
               label=f"best val epoch {best_ep+1}")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.set_title("Training curve (simple baseline)")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig("./training_curve_simple.png", dpi=150)
    plt.show()

## Evaluation 

Original source:  NaviTrace evaluation notebook (leggedrobotics/navitrace) 

In [ ]:
@functools.lru_cache(maxsize=4)
def penalty_lookup(embodiment):
    df  = pd.read_csv(CFG["penalty_tsv"], sep="\t")
    with open(CFG["m2f_config"]) as f:
        m2f = json.load(f)
    id2label = {int(k): v for k, v in m2f["id2label"].items()}
    lkp = {}
    for lid, name in id2label.items():
        row = df[df["category"] == name]
        lkp[lid] = float(row.iloc[0][embodiment]) * 0.8 if len(row) else 0.0
    return lkp

def rasterize(trace, H, W):
    pts, pixels = np.array(trace), []
    if len(pts) > 1:
        for i in range(len(pts)-1):
            r0,c0=int(round(pts[i][1])),int(round(pts[i][0]))
            r1,c1=int(round(pts[i+1][1])),int(round(pts[i+1][0]))
            rr,cc,_=line_aa(r0,c0,r1,c1)
            v=(rr>=0)&(rr<H)&(cc>=0)&(cc<W)
            pixels.extend(zip(rr[v],cc[v]))
    elif len(pts)==1:
        r,c=int(round(pts[0][1])),int(round(pts[0][0]))
        if 0<=r<H and 0<=c<W: pixels.append((r,c))
    return np.array(pixels)

def penalty_mask(seg, gt_px, emb, dthr=35):
    H,W=seg.shape; mask=np.zeros((H,W),dtype=float)
    lkp=penalty_lookup(emb); gtp=rasterize(gt_px,H,W)
    if len(gtp)==0: return mask
    tree=KDTree(gtp); ids=seg.ravel()
    und=np.where(np.isin(ids,list(lkp.keys())))[0]
    if und.size==0: return mask
    rows,cols=np.unravel_index(und,(H,W))
    coords=np.vstack((rows,cols)).T
    dist,_=tree.query(coords); pen_idx=coords[dist>dthr]
    if pen_idx.size>0:
        rp,cp=pen_idx[:,0],pen_idx[:,1]
        mask[rp,cp]=np.vectorize(lkp.get)(seg[rp,cp],0)
    return mask

def sem_penalty(pred_px, pmask):
    H,W=pmask.shape; vals=[]
    for i in range(len(pred_px)-1):
        x1,y1=int(round(pred_px[i][0])),int(round(pred_px[i][1]))
        x2,y2=int(round(pred_px[i+1][0])),int(round(pred_px[i+1][1]))
        rr,cc=sk_line(y1,x1,y2,x2)
        v=(rr>=0)&(rr<H)&(cc>=0)&(cc<W)
        vals.extend(pmask[rr[v],cc[v]].tolist())
    return float(np.mean(vals)) if vals else 0.0

def calc_fde(p,g):
    return float(np.linalg.norm(np.array(p[-1])-np.array(g[-1])))

def calc_dtw(p,g):
    p=np.array(p,dtype=float); g=np.array(g,dtype=float)
    if len(p)!=len(g):
        longer,shorter=(p,g) if len(p)>=len(g) else (g,p)
        d=np.cumsum([0]+[np.linalg.norm(shorter[i]-shorter[i-1])
                         for i in range(1,len(shorter))])
        tot=d[-1]
        if tot>0: d=d/tot
        t=np.linspace(0,1,len(longer))
        shorter=np.column_stack([np.interp(t,d,shorter[:,0]),
                                  np.interp(t,d,shorter[:,1])])
        p,g=(longer,shorter) if len(p)>=len(g) else (shorter,longer)
    n,m=len(p),len(g); D=np.full((n+1,m+1),np.inf); D[0,0]=0
    for i in range(1,n+1):
        for j in range(1,m+1):
            D[i,j]=np.linalg.norm(p[i-1]-g[j-1])+min(D[i-1,j],D[i,j-1],D[i-1,j-1])
    return float(D[n,m])

def norm_score(raw):
    return (CFG["bad_score_threshold"]-raw)/CFG["bad_score_threshold"]*100

In [ ]:
model.eval()
with torch.no_grad():
    goal_raw, trace_norm = model(val_joint.to(DEVICE))
    goal_raw   = goal_raw.cpu().numpy()
    trace_norm = trace_norm.cpu().numpy()

# Denormalize goal predictions
goal_preds = np.stack([denorm_goal(g) for g in goal_raw])

print(f"Goal  predictions : {goal_preds.shape}")
print(f"Trace predictions : {trace_norm.shape}")
print(f"Goal x mean={goal_preds[:,0].mean():.3f}  std={goal_preds[:,0].std():.3f}")
print(f"Goal y mean={goal_preds[:,1].mean():.3f}  std={goal_preds[:,1].std():.3f}")

In [ ]:
nt_scores=[]; dtw_scores=[]; fde_scores=[]; pen_scores=[]
skipped=0

for i in tqdm(range(len(val_samples)), desc="Evaluating"):
    sample    = val_samples[i]
    gt_traces = sample["ground_truth"].get(CFG["embodiment"])
    if gt_traces is None or len(gt_traces)==0:
        skipped+=1; continue

    img=sample["image"]; orig_w,orig_h=img.size
    seg=np.array(sample["segmentation_mask"])
    trace_px=pred_padded_to_pixel(trace_norm[i], orig_w, orig_h)

    best_raw=best_d=best_f=best_p=float("inf")
    for gt_px in gt_traces:
        pmask=penalty_mask(seg,gt_px,CFG["embodiment"],CFG["penalty_dist_thr"])
        p_val=sem_penalty(trace_px,pmask)
        f_val=calc_fde(trace_px,gt_px)
        d_val=calc_dtw(trace_px,gt_px)
        raw=d_val+f_val+p_val
        if raw<best_raw:
            best_raw,best_d,best_f,best_p=raw,d_val,f_val,p_val

    nt_scores.append(norm_score(best_raw))
    dtw_scores.append(best_d); fde_scores.append(best_f); pen_scores.append(best_p)

print(f" Evaluated={len(nt_scores)}  Skipped={skipped}")
print(f" Trace score  : {np.mean(nt_scores):.2f} +/ - {np.std(nt_scores):.2f}")
print(f" Mean DTW (px): {np.mean(dtw_scores):.2f}")
print(f" Mean FDE (px): {np.mean(fde_scores):.2f}")
print(f" Mean penalty : {np.mean(pen_scores):.2f}")

## Eval for leaderboard 

In [ ]:
if "dataset" not in dir():
    dataset = load_dataset("leggedrobotics/navitrace")

test_samples = []
for s in tqdm(list(dataset["test"]), desc="Filtering test"):
    if CFG["embodiment"] in s.get("embodiments", []):
        test_samples.append(s)
print(f"Test samples : {len(test_samples)}")

In [ ]:
if "clip_model" not in dir():
    clip_model, clip_preprocess = clip.load(CFG["clip_model"], device=DEVICE)
    clip_model.eval()
    for p in clip_model.parameters():
        p.requires_grad = False

class TestDS(Dataset):
    def __init__(self, samps):
        self.samps = samps
    def __len__(self): return len(self.samps)
    def __getitem__(self, idx):
        s = self.samps[idx]
        img = s["image"].convert("RGB")
        padded, _ = pad_to_square(img)
        return clip_preprocess(padded), clip.tokenize([s["task"]], truncate=True)[0]

@torch.no_grad()
def extract_test(samps):
    ds  = TestDS(samps)
    ldr = DataLoader(ds, batch_size=64, shuffle=False, num_workers=0)
    imgs, txts = [], []
    for i_t, t_t in tqdm(ldr, desc="Extracting test"):
        i_f = F.normalize(clip_model.encode_image(i_t.to(DEVICE)).float(), dim=-1)
        t_f = F.normalize(clip_model.encode_text(t_t.to(DEVICE)).float(),  dim=-1)
        imgs.append(i_f.cpu()); txts.append(t_f.cpu())
    return torch.cat(imgs), torch.cat(txts)

test_img_emb, test_txt_emb = extract_test(test_samples)
test_joint = torch.cat([test_img_emb, test_txt_emb], dim=-1)

model.eval()
with torch.no_grad():
    test_goal_raw, test_trace_norm = model(test_joint.to(DEVICE))
    test_goal_raw   = test_goal_raw.cpu().numpy()
    test_trace_norm = test_trace_norm.cpu().numpy()

test_goal_preds = np.stack([denorm_goal(g) for g in test_goal_raw])

results = []
for i, sample in enumerate(test_samples):
    img            = sample["image"]
    orig_w, orig_h = img.size
    pred_px        = pred_padded_to_pixel(test_trace_norm[i], orig_w, orig_h)
    results.append({
        "sample_id":  sample["sample_id"],
        "embodiment": CFG["embodiment"],
        "category":  sample["category"],
        "prediction": pred_px,
    })

pd.DataFrame(results).to_csv("./test_predictions_simple.tsv", sep="\t", index=False) 